# ChargebackOps Merchant Agent - outcome-based RL on Qwen2.5-3B (fp16 LoRA)

End-to-end pipeline for a single Colab T4 (or Kaggle T4):

1. **Phase A - JSON SFT** on heuristic rollouts. Teaches the model the env action schema.
2. **Phase B - GRPO with outcome reward**. Reward = terminal `$` PnL after the model's action plus heuristic tail-rollout. The merchant policy is pushed toward actions that *win money* against the scripted Issuer + arbitration, not actions that match the heuristic. This is real RLVR (verifiable rewards) - the verifier is the dispute outcome.
3. **Eval** every checkpoint against the heuristic baseline + naive baseline.

**Why outcome reward over heuristic-match**: heuristic-match training is supervised distillation in disguise - the model can never beat the teacher and the reward is gameable by mimicry. Outcome reward is dollar-denominated, adversarially-verified by the Issuer, and ungameable: the only way to earn it is to actually win disputes.

**Theme alignment**: Multi-Agent (merchant vs scripted Issuer) primary; Professional Tasks (B2B chargeback workflow) and Long-Horizon (multi-round arbitration) secondary.

Model: `Qwen/Qwen2.5-3B-Instruct` fp16 + LoRA r=16. Fits T4 with no bitsandbytes.

## 0. Setup - install deps + clone repo

In [ ]:
# GPU + repo setup for Colab T4 (also runs on Kaggle T4).
# Pin the training stack to the combo that supports both:
#   - SFTTrainer.compute_loss without the shape-bug regression
#   - vanilla GRPOTrainer with reward_funcs returning per-completion floats
# Newer Colab/Kaggle images preinstall transformers 5.x + hub 1.x; this cell
# isolates a self-consistent training stack in a private deps directory.
import os
import shutil
import subprocess
import sys
import importlib

if os.path.isdir('/content'):
    WORK_DIR = '/content'
elif os.path.isdir('/kaggle/working'):
    WORK_DIR = '/kaggle/working'
else:
    raise RuntimeError('Notebook expects Colab or Kaggle. Set WORK_DIR manually otherwise.')
os.chdir(WORK_DIR)
DRIVE_ROOT = '/content/drive/MyDrive'
PERSIST_ROOT = (
    os.path.join(DRIVE_ROOT, 'chargebackops-artifacts')
    if os.path.isdir(DRIVE_ROOT) else WORK_DIR
)
os.makedirs(PERSIST_ROOT, exist_ok=True)
print('work dir:', WORK_DIR)
print('artifact root:', PERSIST_ROOT)
print(subprocess.check_output(['nvidia-smi', '-L']).decode())

PYTHON = sys.executable
PIP = [PYTHON, '-m', 'pip']
DEPS_DIR = os.path.join(WORK_DIR, '_pydeps')
if os.path.isdir(DEPS_DIR):
    shutil.rmtree(DEPS_DIR)
os.makedirs(DEPS_DIR, exist_ok=True)
print('python:', PYTHON)
print('pip cmd:', ' '.join(PIP))
print('deps dir:', DEPS_DIR)

# STEP 1 - torch trio for cu128. Turing T4 supports cu118+; cu128 wheels work.
subprocess.run(
    PIP + ['install', '-q', '--no-cache-dir',
           'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
           '--index-url', 'https://download.pytorch.org/whl/cu128'],
    check=True, cwd=WORK_DIR,
)

CORE_STACK = [
    'transformers==4.51.3',
    'trl==0.21.0',
    'peft==0.14.0',
    'accelerate==1.0.1',
    'tokenizers==0.21.4',
    'huggingface-hub==0.30.2',
]

# STEP 2a - install the exact Hugging Face training stack into a private deps
# directory. This avoids Colab/Kaggle system packages shadowing pinned wheels.
# transformers 4.51.3 requires huggingface-hub>=0.30,<1.0, so 0.30.2 is a
# consistent lower-end hub pin for this stack.
subprocess.run(
    PIP + ['install', '-q', '--no-cache-dir', '--upgrade', '--target', DEPS_DIR, '--no-deps'] + CORE_STACK,
    check=True, cwd=WORK_DIR,
)

# STEP 2b - supporting libs into the same private deps dir. This can still drag
# newer transitive HF packages into DEPS_DIR, so the core stack is cleaned and
# re-applied immediately afterward.
subprocess.run(
    PIP + ['install', '-q', '--upgrade', '--target', DEPS_DIR, '--upgrade-strategy=only-if-needed',
           'datasets>=2.20,<4.0',
           'matplotlib>=3.8',
           'pydantic>=2.10',
           'openenv-core>=0.2.2'],
    check=True, cwd=WORK_DIR,
)

# STEP 2c - remove any shadowing core-package copies added by supporting deps,
# then reinstall the exact training stack into DEPS_DIR.
CORE_TOPLEVEL = {'transformers', 'tokenizers', 'trl', 'peft', 'accelerate', 'huggingface_hub'}
for name in list(os.listdir(DEPS_DIR)):
    stem = name.split('-')[0]
    if stem in CORE_TOPLEVEL:
        path = os.path.join(DEPS_DIR, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
subprocess.run(
    PIP + ['install', '-q', '--no-cache-dir', '--upgrade', '--target', DEPS_DIR, '--no-deps'] + CORE_STACK,
    check=True, cwd=WORK_DIR,
)

# Make the private deps directory shadow system site-packages for this kernel
# and for any child Python processes started later.
os.environ['PYTHONPATH'] = DEPS_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')
if DEPS_DIR not in sys.path:
    sys.path.insert(0, DEPS_DIR)
importlib.invalidate_caches()
for mod in list(sys.modules):
    if mod.split('.')[0] in {'huggingface_hub', 'transformers', 'tokenizers', 'trl', 'peft', 'accelerate', 'datasets'}:
        del sys.modules[mod]

# Clone repo (always fresh) so the editable install matches main.
REPO_DIR = os.path.join(WORK_DIR, 'chargebackops')
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/MitudruDutta/ChargeBackOps.git', REPO_DIR],
    check=True, cwd=WORK_DIR,
)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

# Editable install with --no-deps so pyproject's app/server deps do not alter
# the training stack we just pinned.
subprocess.run(PIP + ['install', '-q', '-e', '.', '--no-deps'],
               check=True, cwd=REPO_DIR)

# Verify the runtime imports are coming from the private deps directory.
import importlib.metadata as md_
PINNED_DIST_NAMES = {'transformers', 'tokenizers', 'trl', 'peft', 'accelerate', 'huggingface-hub'}
_original_md_version = md_.version

def _deps_dir_version(pkg_name):
    normalized = pkg_name.lower().replace('_', '-')
    if normalized in PINNED_DIST_NAMES:
        for dist in md_.distributions(path=[DEPS_DIR]):
            if (dist.metadata.get('Name') or '').lower() == normalized:
                return dist.version
    return _original_md_version(pkg_name)

md_.version = _deps_dir_version
import importlib.metadata
importlib.metadata.version = _deps_dir_version

import huggingface_hub
import transformers
import tokenizers
import trl
import peft
import accelerate

print('torch           ', md_.version('torch'))
print('torchvision     ', md_.version('torchvision'))
print('transformers    ', transformers.__version__, transformers.__file__)
print('tokenizers      ', tokenizers.__version__, tokenizers.__file__)
print('huggingface_hub ', huggingface_hub.__version__, huggingface_hub.__file__)
print('trl             ', trl.__version__, trl.__file__)
print('peft            ', peft.__version__, peft.__file__)
print('accelerate      ', accelerate.__version__, accelerate.__file__)
print('openenv-core    ', md_.version('openenv-core'))
assert transformers.__version__ == '4.51.3', 'transformers pin failed'
assert tokenizers.__version__.startswith('0.21'), 'tokenizers pin failed'
assert huggingface_hub.__version__ == '0.30.2', 'huggingface_hub runtime pin failed'
assert trl.__version__ == '0.21.0', 'trl pin failed'
assert peft.__version__ == '0.14.0', 'peft pin failed'
assert accelerate.__version__ == '1.0.1', 'accelerate pin failed'
assert os.path.realpath(DEPS_DIR) in os.path.realpath(huggingface_hub.__file__), 'huggingface_hub not loaded from DEPS_DIR'


In [ ]:
# Path + module-cache flush so the editable install resolves before any other import.
import os, sys, importlib, logging, torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Prefer the notebook-local pinned deps dir over any Colab/Kaggle system wheels.
DEPS_DIR = globals().get('DEPS_DIR') or os.path.join('/content', '_pydeps')
if os.path.isdir(DEPS_DIR) and DEPS_DIR not in sys.path:
    sys.path.insert(0, DEPS_DIR)

# Silence transformers per-layer "Caching is incompatible..." spam at scale
# (every GRPO rollout fires it once per layer otherwise, hiding loss logs).
import transformers
transformers.logging.set_verbosity_error()
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('transformers.models.qwen2.modeling_qwen2').setLevel(logging.ERROR)

REPO_DIR = globals().get('REPO_DIR') or (
    '/content/chargebackops' if os.path.isdir('/content/chargebackops')
    else '/kaggle/working/chargebackops'
)
PERSIST_ROOT = globals().get('PERSIST_ROOT') or (
    os.path.join('/content/drive/MyDrive', 'chargebackops-artifacts')
    if os.path.isdir('/content/drive/MyDrive') else os.path.dirname(REPO_DIR)
)
os.makedirs(PERSIST_ROOT, exist_ok=True)
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Repository checkout missing at {REPO_DIR}. Run setup first.')
sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for mod in list(sys.modules):
    if mod.startswith(('scenarios', 'training', 'evaluation', 'server', 'core', 'runners', 'connectors')):
        del sys.modules[mod]
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('repo:', REPO_DIR)
print('deps dir:', DEPS_DIR)
print('artifact root:', PERSIST_ROOT)

## 1. Load Qwen2.5-3B-Instruct fp16 + attach LoRA

* `dtype=torch.float16` - T4 = Turing sm_75, no bf16 hardware.
* LoRA r=16 on q/k/v/o + gate/up/down. ~9M trainable params.
* `gradient_checkpointing` + `enable_input_require_grads` keeps activations off VRAM during backward.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-3B-Instruct')
print('MODEL_ID:', MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # required for GRPO generation

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

lora_target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj']
lora_rank = 16
lora_alpha = 32

lora_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | '
      f'free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

## 2. Phase A - SFT on heuristic rollouts

Builds (prompt, oracle_completion) pairs by rolling the scripted heuristic on every headline + generated task. Wraps in the Qwen chat template so the model learns the same prompt format used at inference. After this phase the model emits valid JSON with the right `action_type` per state - solves the *always emit `select_case`* collapse before GRPO ever runs.

In [ ]:
from datasets import Dataset
from scenarios.simulation import list_tasks, get_task
from training.sft_dataset import build_sft_dataset
from collections import Counter

# Synthetic pool. Default 10k rows for T4 speed; override via env var.
SFT_TARGET_ROWS = int(os.environ.get('SFT_TARGET_ROWS', '4000'))
SFT_MAX_ROWS = int(os.environ.get('SFT_MAX_ROWS', str(SFT_TARGET_ROWS)))
SFT_SEED_START = int(os.environ.get('SFT_SEED_START', '1000'))
SFT_SEED_BATCH = int(os.environ.get('SFT_SEED_BATCH', '128'))
SFT_MAX_STATES_PER_TASK = int(os.environ.get('SFT_MAX_STATES_PER_TASK', '24'))
GRPO_SEED_COUNT = int(os.environ.get('GRPO_SEED_COUNT', '160'))

# Holdout seeds excluded from training so eval is defensible.
HOLDOUT_SEEDS_BY_DIFF = {
    'easy': {42},
    'medium': {17, 99},
    'hard': {7, 53},
    'nightmare': {31, 77},
}
DIFFICULTIES = ['easy', 'medium', 'hard', 'nightmare']

headline_task_ids = [t.task_id for t in list_tasks()]
task_ids = list(headline_task_ids)
raw_sft = build_sft_dataset(headline_task_ids, max_states_per_task=SFT_MAX_STATES_PER_TASK)
generated_train_task_ids = []

seed_cursor = SFT_SEED_START
while len(raw_sft) < SFT_TARGET_ROWS:
    batch_task_ids = []
    for diff in DIFFICULTIES:
        blocked = HOLDOUT_SEEDS_BY_DIFF.get(diff, set())
        for seed in range(seed_cursor, seed_cursor + SFT_SEED_BATCH):
            if seed in blocked:
                continue
            tid = f'generated_{diff}_s{seed}'
            get_task(tid)
            batch_task_ids.append(tid)
    raw_sft.extend(build_sft_dataset(batch_task_ids, max_states_per_task=SFT_MAX_STATES_PER_TASK))
    generated_train_task_ids.extend(batch_task_ids)
    task_ids.extend(batch_task_ids)
    seed_cursor += SFT_SEED_BATCH
    print(f'generated SFT rows: {len(raw_sft):,} / target {SFT_TARGET_ROWS:,}')

if len(raw_sft) > SFT_MAX_ROWS:
    raw_sft = raw_sft[:SFT_MAX_ROWS]

# Seed list for GRPO state-action curriculum (smaller than SFT pool because
# GRPO rollouts are slower than SFT forward passes).
seeds = list(range(SFT_SEED_START, SFT_SEED_START + GRPO_SEED_COUNT))

def to_chat_text(prompt, completion):
    return tokenizer.apply_chat_template(
        [
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': completion},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

sft_rows = [{'text': to_chat_text(s['prompt'], s['completion'])} for s in raw_sft]
sft_dataset = Dataset.from_list(sft_rows)

atype_counts = Counter(s['action_type'] for s in raw_sft)
print(f'SFT samples: {len(sft_dataset):,}, unique tasks: {len(set(s["task_id"] for s in raw_sft)):,}')
print(f'headline tasks: {len(headline_task_ids)}, generated train tasks used: {len(generated_train_task_ids):,}')
print(f'excluded generated holdout seeds: {HOLDOUT_SEEDS_BY_DIFF}')
print(f'action_type distribution: {dict(atype_counts)}')
print('sample (first 500 chars):')
print(sft_rows[0]['text'][:500])

In [ ]:
from trl import SFTConfig, SFTTrainer

OUT_ROOT = PERSIST_ROOT
SFT_DIR = os.path.join(OUT_ROOT, 'sft-merchant-agent')
GRPO_DIR = os.path.join(OUT_ROOT, 'grpo-merchant-agent')

SFT_FINAL_DIR = os.path.join(SFT_DIR, 'final')
RUN_SFT_TRAIN = os.environ.get('RUN_SFT_TRAIN', 'auto').strip().lower()
TRAIN_SFT = RUN_SFT_TRAIN in {'1', 'true', 'yes', 'y', 'on'} or (
    RUN_SFT_TRAIN == 'auto' and not os.path.isdir(SFT_FINAL_DIR)
)
SFT_EPOCHS = float(os.environ.get('SFT_EPOCHS', '1'))
SFT_LR = float(os.environ.get('SFT_LR', '1e-4'))
SFT_MAX_STEPS = int(os.environ.get('SFT_MAX_STEPS', '300'))

if not TRAIN_SFT:
    print(f'Skipping SFT train; using existing adapter at {SFT_FINAL_DIR}')
else:
    sft_config = SFTConfig(
        output_dir=SFT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=SFT_EPOCHS,
        max_steps=SFT_MAX_STEPS,
        learning_rate=SFT_LR,
        logging_steps=10,
        save_steps=300,
        save_total_limit=2,
        bf16=False,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        max_length=1024,
        dataset_text_field='text',
        report_to='none',
        optim='adamw_torch',
        warmup_ratio=0.03,
    )
    print(f'SFT config: rows={len(sft_dataset):,}, epochs={SFT_EPOCHS}, lr={SFT_LR}, max_steps={SFT_MAX_STEPS}')
    if hasattr(model, 'config'):
        model.config.use_cache = False
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )
    sft_trainer.train()
    sft_trainer.save_model(SFT_FINAL_DIR)
    del sft_trainer
    torch.cuda.empty_cache()
    print(f'PEAK VRAM (SFT): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 2.5. Merge SFT LoRA into base, attach fresh LoRA for GRPO

`accelerate.unwrap_model_for_generation` calls `merge_adapter()` + `unmerge_adapter()` around generation. With fp16 LoRA the round-trip can lose enough precision that completions degrade. Fix: bake SFT into base via `merge_and_unload()`, then attach a fresh zero-initialized LoRA. The fresh adapter starts as identity, so generation emits SFT-quality output regardless of TRL adapter toggling. GRPO then trains the fresh adapter on top.

In [ ]:
# Reload saved SFT LoRA into a fresh base, merge it, then attach a fresh Phase B LoRA.
from peft import PeftModel
import gc

if not os.path.isdir(SFT_FINAL_DIR):
    raise FileNotFoundError(f'Missing SFT adapter: {SFT_FINAL_DIR}. Run Phase A or upload the adapter first.')

for name in ['model', 'base_model', 'merged_base']:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print(f'before SFT reload: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
sft_model = PeftModel.from_pretrained(fresh_base, SFT_FINAL_DIR)
merged_base = sft_model.merge_and_unload()
del sft_model, fresh_base
gc.collect()
torch.cuda.empty_cache()
print(f'after SFT merge: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')
merged_base.enable_input_require_grads()

# Sanity: SFT-baked base should emit clean JSON deterministically.
from training.env_adapter import build_prompt
from server.chargeback_ops_environment import ChargebackOpsEnvironment
env = ChargebackOpsEnvironment()
obs = env.reset(task_id='goods_not_received_easy')
chat = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': build_prompt(obs.model_dump())}],
    tokenize=False, add_generation_prompt=True,
)
inp = tokenizer(chat, return_tensors='pt').to(merged_base.device)
merged_base.eval()
with torch.no_grad():
    out = merged_base.generate(
        **inp,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print('merged-base gen:', repr(tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=False)))
merged_base.train()

# Attach fresh Phase B LoRA. lora_dropout=0.1 keeps stochasticity ALIVE during
# train()-mode generation. The v1 attempt set this to 0 + low temp + small
# num_generations - result was every group of 4 emitted identical completions,
# std=0, GRPO advantage=0, no learning. With dropout=0.1 plus temp=1.3 +
# num_generations=8 set in the GRPO cell, within-group variance is restored.
lora_phase_b = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(merged_base, lora_phase_b)
model.enable_input_require_grads()
model.print_trainable_parameters()
print(f'after fresh Phase B LoRA: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')


## 3. Phase B - GRPO with outcome reward (RLVR, not distillation)

Reward source: terminal `$` PnL after the model's action plus heuristic tail-rollout. The merchant earns positive reward only when its action leads to a winning packet against the scripted Issuer or arbitration. A second format-shaping reward provides dense early-training signal so GRPO has gradient before the policy can produce winning packets.

It optimizes "win the dispute." The Issuer + arbitration are the verifier - they cannot be tricked because reward is dollar-denominated outcome.

In [ ]:
from training.reward_adapter import build_state_action_dataset

# State-action samples: (task_id, state_step, prompt) tuples captured by
# rolling the heuristic forward on each task. The model will be asked to
# pick the next action at each captured state.
PHASE_B_MAX_STATES_PER_TASK = int(os.environ.get('PHASE_B_MAX_STATES_PER_TASK', '10'))
GRPO_DIFFICULTIES = tuple(
    d.strip()
    for d in os.environ.get('GRPO_DIFFICULTIES', 'easy,medium,hard,nightmare').split(',')
    if d.strip()
)
curriculum_task_ids = [
    t.task_id for t in list_tasks()
    if t.difficulty in GRPO_DIFFICULTIES
]
for diff in GRPO_DIFFICULTIES:
    blocked = HOLDOUT_SEEDS_BY_DIFF.get(diff, set())
    for s in seeds:
        if s in blocked:
            continue
        tid = f'generated_{diff}_s{s}'
        try:
            get_task(tid)
            if tid not in curriculum_task_ids:
                curriculum_task_ids.append(tid)
        except Exception:
            pass

raw_grpo = build_state_action_dataset(
    curriculum_task_ids, max_states_per_task=PHASE_B_MAX_STATES_PER_TASK,
)

def to_chat_prompt(prompt):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False, add_generation_prompt=True,
    )

grpo_rows = []
for sample in raw_grpo:
    chat_prompt = to_chat_prompt(sample['prompt'])
    n_tokens = len(tokenizer(chat_prompt, add_special_tokens=False)['input_ids'])
    if n_tokens <= 1000:
        grpo_rows.append({
            'prompt': chat_prompt,
            'task_id': sample['task_id'],
            'state_step': int(sample['state_step']),
        })

grpo_dataset = Dataset.from_list(grpo_rows)
unique_tasks = len({row['task_id'] for row in grpo_rows})
print(f'GRPO state-action samples: {len(grpo_dataset)}, '
      f'unique tasks: {unique_tasks}, difficulties={GRPO_DIFFICULTIES}, '
      f'max_states_per_task={PHASE_B_MAX_STATES_PER_TASK}')

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.outcome_reward import compute_outcome_reward, compute_format_reward

# Re-arm hooks and disable cache. Do NOT zero out dropout - the merge cell
# attached the Phase B LoRA with lora_dropout=0.1 specifically so train()-mode
# generation has stochasticity. Zeroing it here was the v1 bug.
model.enable_input_require_grads()
if hasattr(model, 'config'):
    model.config.use_cache = False
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model, 'generation_config'):
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id

def outcome_reward_fn(prompts, completions, **kwargs):
    task_ids = kwargs.get('task_id') or kwargs.get('task_ids')
    state_steps = kwargs.get('state_step') or kwargs.get('state_steps')
    return compute_outcome_reward(
        prompts, completions,
        task_ids=task_ids, state_steps=state_steps,
    )

def format_reward_fn(prompts, completions, **kwargs):
    return compute_format_reward(prompts, completions)

# CRITICAL sampling kwargs - rewritten after the v1 run had grad_norm=0.0 on
# 95% of steps. The v1 logs showed:
#   - frac_reward_zero_std=1.0 on ~80% of steps (all 4 generations identical)
#   - entropy=0.001-0.017 (policy near-delta after SFT mean_acc=0.96)
#   - When std=0 inside a group, advantage=0 and gradient=0.
#
# Fix: aggressively widen the sampling distribution.
#   temperature: 0.7 -> 1.3   (past 1.0 breaks the SFT argmax lock)
#   top_p:       0.9 -> 1.0   (no nucleus truncation)
#   top_k:       50  -> 0     (no top-k truncation)
#   num_generations: 4 -> 6   (1.5x within-group variance odds, T4-friendly)
#   learning_rate: 5e-6 -> 2e-5 (bigger push to escape SFT collapse)
#   beta:        0.0 -> 0.04  (small KL anchor; v1 collapse risk is gone)
#   lora_dropout: 0.0 -> 0.1  (set in the merge cell, kept here)
#
# The format_reward_fn (-0.10 for invalid JSON) is the safety net that stops
# the model from drifting into pure noise at the higher temperature.
grpo_config = GRPOConfig(
    output_dir=GRPO_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=6,
    max_prompt_length=1024,
    max_completion_length=192,
    learning_rate=float(os.environ.get('GRPO_LR', '2e-5')),
    max_steps=int(os.environ.get('GRPO_MAX_STEPS', '60')),
    logging_steps=5,
    save_steps=60,
    save_total_limit=2,
    bf16=False,
    fp16=True,
    max_grad_norm=0.5,
    gradient_checkpointing=False,
    report_to='none',
    beta=0.04,
    temperature=1.3,
    top_p=1.0,
    top_k=0,
    repetition_penalty=1.0,
    use_vllm=False,
    log_completions=True,
    num_completions_to_print=2,
    optim='adamw_torch',
    lr_scheduler_type='constant',
)

RUN_GRPO = os.environ.get('RUN_GRPO', '1').strip().lower() not in {'0', 'false', 'no'}
if RUN_GRPO:
    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[outcome_reward_fn, format_reward_fn],
        args=grpo_config,
        train_dataset=grpo_dataset,
    )
    grpo_trainer.train()
    grpo_trainer.save_model(os.path.join(GRPO_DIR, 'final'))
    del grpo_trainer
    torch.cuda.empty_cache()
else:
    print('RUN_GRPO=0: skipped GRPO training')
print(f'PEAK VRAM (GRPO): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')


## 4. Per-checkpoint eval - overall + per-difficulty

Loads each saved adapter, plays full episodes across the headline catalog, and plots the curve. Stall detection in `run_episode_with_text_policy` ensures degenerate checkpoints reach grading instead of returning 0.

In [ ]:
import glob, re, gc
from peft import PeftModel
from training.curve import (
    evaluate_checkpoint, evaluate_checkpoint_by_family,
    plot_training_curve, plot_training_curve_by_family,
)

# Free EVERYTHING from training cells before loading eval models. The GRPO
# trainer's model/merged_base/sft_model are still pinning 6+GB of VRAM otherwise.
for name in ['model', 'merged_base', 'base_model', 'sft_model', 'sft_trainer',
            'grpo_trainer', 'fresh_base', 'tmp_base', 'sft_for_merge']:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print(f'before eval setup: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Load BASE once. For 'base' and 'sft' eval, attach/detach SFT adapter on this.
eval_tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if eval_tok.pad_token is None:
    eval_tok.pad_token = eval_tok.eos_token
eval_tok.padding_side = 'left'

eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True,
)
eval_base.eval()
print(f'after eval_base load: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')

# For GRPO eval we need a SECOND base with SFT pre-merged. Load it once, hold
# in VRAM. With 3B fp16 we need ~12-13 GB for both bases - fits T4 but tight.
sft_merged_base = None
if os.path.isdir(SFT_FINAL_DIR):
    tmp_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True,
    )
    sft_for_merge = PeftModel.from_pretrained(tmp_base, SFT_FINAL_DIR)
    sft_merged_base = sft_for_merge.merge_and_unload()
    sft_merged_base.eval()
    del sft_for_merge, tmp_base
    gc.collect(); torch.cuda.empty_cache()
    print(f'after sft_merged_base: VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')


def make_policy(adapter_path, adapter_kind):
    """Return policy fn + cleanup fn. No new base loads. Adapters swap on
    pre-loaded eval_base / sft_merged_base."""
    if adapter_kind == 'grpo':
        if sft_merged_base is None:
            raise RuntimeError('grpo eval needs SFT adapter merged first')
        target = sft_merged_base
    else:
        target = eval_base

    if adapter_kind == 'base' or adapter_path is None:
        m = target
        cleanup = lambda: None
    else:
        adapter_name = f'eval_{adapter_kind}_{abs(hash(adapter_path))}'
        m = PeftModel.from_pretrained(target, adapter_path, adapter_name=adapter_name)
        m.eval()
        def cleanup(_m=m, _name=adapter_name):
            try:
                _m.delete_adapter(_name)
            except Exception:
                pass
            gc.collect(); torch.cuda.empty_cache()

    def policy(prompt):
        chat = eval_tok.apply_chat_template(
            [{'role': 'user', 'content': prompt}],
            tokenize=False, add_generation_prompt=True,
        )
        inputs = eval_tok(chat, return_tensors='pt', truncation=True, max_length=1024).to(m.device)
        with torch.no_grad():
            out = m.generate(
                **inputs, max_new_tokens=192, do_sample=False,
                pad_token_id=eval_tok.eos_token_id, eos_token_id=eval_tok.eos_token_id,
            )
        return eval_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return policy, cleanup


# Catalog: untrained base, SFT final, every saved GRPO checkpoint + final.
ckpt_specs = [('base', None, 0, 'base'),
              ('sft', SFT_FINAL_DIR, 1, 'sft')]
grpo_dirs = sorted(
    glob.glob(os.path.join(GRPO_DIR, 'checkpoint-*')),
    key=lambda p: int(re.search(r'checkpoint-(\d+)', p).group(1)),
)
for d in grpo_dirs:
    step = int(re.search(r'checkpoint-(\d+)', d).group(1))
    ckpt_specs.append((f'grpo-{step}', d, 1 + step, 'grpo'))
final_grpo = os.path.join(GRPO_DIR, 'final')
if os.path.isdir(final_grpo):
    last_step = max(int(re.search(r'checkpoint-(\d+)', d).group(1)) for d in grpo_dirs) if grpo_dirs else 0
    ckpt_specs.append(('grpo-final', final_grpo, 1 + last_step + 1, 'grpo'))

overall = []
grouped = []
for label, path, step, kind in ckpt_specs:
    print(f'eval {label} from {path}')
    pol, cleanup = make_policy(path, kind)
    overall.append(evaluate_checkpoint(step=step, policy=pol))
    grouped.append(evaluate_checkpoint_by_family(step=step, policy=pol))
    cleanup()

print('\nOVERALL CURVE:')
for c in overall:
    print(f'  step={c.step:4d} mean={c.mean_score:.4f}')

print('\nPER-FAMILY CURVE:')
for g in grouped:
    line = f'  step={g.step:4d}'
    for fam in sorted(g.by_family.keys()):
        line += f'  {fam}={g.by_family[fam].mean_score:.3f}'
    print(line)

from runners.benchmark_runner import run_policy_sweep
sweep = run_policy_sweep()
heur_overall = next(s.mean_score for s in sweep.policies if s.policy == 'heuristic')

FIG_DIR = os.path.join(REPO_DIR, 'docs', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
plot_training_curve(
    overall, os.path.join(FIG_DIR, 'training_curve.png'),
    baseline_scores={'heuristic': heur_overall, 'naive': 0.0},
)
plot_training_curve_by_family(
    grouped, os.path.join(FIG_DIR, 'training_curve_by_family.png'),
    family_order=['easy', 'medium', 'hard', 'nightmare'],
)
print(f'\nfigures saved to {FIG_DIR}/')


## 5. Diagnose final checkpoint

Print the trained checkpoint's completion vs. the heuristic oracle on three representative tasks (easy, hard, nightmare). Verifies the model emits valid JSON and shows what the trained policy actually does.

In [ ]:
from training.env_adapter import build_prompt, parse_completion
from training.outcome_reward import compute_outcome_reward
from server.chargeback_ops_environment import ChargebackOpsEnvironment
from runners.benchmark_runner import heuristic_policy

final_adapter = (
    os.path.join(GRPO_DIR, 'final')
    if os.path.isdir(os.path.join(GRPO_DIR, 'final'))
    else (grpo_dirs[-1] if grpo_dirs else SFT_FINAL_DIR)
)
final_kind = 'grpo' if 'grpo-merchant-agent' in final_adapter else 'sft'
print(f'diagnose adapter: {final_adapter}')
policy, m = make_text_policy(final_adapter, final_kind)

for tid in ['goods_not_received_easy', 'queue_optimization_hard', 'generated_nightmare_s31']:
    env = ChargebackOpsEnvironment()
    obs = env.reset(task_id=tid)
    raw = build_prompt(obs.model_dump())
    completion = policy(raw)
    parsed = parse_completion(completion)
    oracle = heuristic_policy(obs.model_dump())
    pnl = compute_outcome_reward(['x'], [completion], task_ids=[tid], state_steps=[0])[0]
    print(f'\n=== {tid} ===')
    print(f'oracle: {oracle.action_type} case={oracle.case_id}')
    print(f'completion (first 200): {repr(completion[:200])}')
    print(f'parsed: {parsed}')
    print(f'outcome PnL (normalized): {pnl:+.3f}')
del m
torch.cuda.empty_cache()

## Done

Artifacts written to `docs/figures/`:
* `training_curve.png` - overall mean rubric score across SFT + GRPO checkpoints, with heuristic baseline.
* `training_curve_by_family.png` - per-difficulty curves (easy / medium / hard / nightmare).

Adapter weights (under `PERSIST_ROOT`):
* `sft-merchant-agent/final/` - Phase A output.
* `grpo-merchant-agent/final/` - Phase B output (GRPO with outcome reward).

To use the trained model:
```python
from peft import PeftModel
sft_model = PeftModel.from_pretrained(base, 'sft-merchant-agent/final')
merged = sft_model.merge_and_unload()
trained = PeftModel.from_pretrained(merged, 'grpo-merchant-agent/final')
```